# 178. BitNet b1.58：Ternary BitLinear、STE 与位打包怎样实现？

> **面试问题：1.58-bit 为什么是三值权重？absmean 量化、激活量化、master weight、整数计算和存储收益怎样验证？**

## 先给结论

`log2(3)≈1.585` 来自三种权重状态 `{-1,0,1}`。BitNet b1.58 训练期保留高精度 master weight，在 forward 用 absmean scale 三值化并通过 STE 回传；激活通常另行低比特量化。理论位宽不会自动变成速度，必须有紧凑打包、专用 GEMM 与端到端测量。

## 推荐回答主线

1. 区分从头量化感知训练与训练后把 FP 模型强行三值化，写出权重 scale 和 ternary code。
2. 实现 activation absmax quant、BitLinear.forward 与 STE，验证梯度更新的是 master weight。
3. 用 base-3 打包/解包证明真实存储合同，并讨论 scale/metadata 开销。
4. 比较数值误差、饱和、吞吐和质量；绑定量化 recipe、布局与 kernel。

## 教学实现边界

教学实现仍调用普通 PyTorch 浮点 matmul 来模拟反量化权重，不产生真实 ternary kernel 加速；位打包仅演示存储编码，不处理 SIMD 对齐和分块 scale。

## 一手资料

- [BitNet](https://arxiv.org/abs/2310.11453)
- [BitNet b1.58](https://arxiv.org/abs/2402.17764)
- [bitnet.cpp](https://arxiv.org/abs/2502.11880)


In [ ]:
import hashlib
import json
import math
from dataclasses import asdict, dataclass

import numpy as np
import warnings
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*", category=FutureWarning)

import torch
from torch import nn

# 小矩阵按每个输出通道的 5 个输入权重分组，便于核对 scale 与整数点积。
torch.manual_seed(178)
GROUP_SIZE = 5
weight = torch.tensor([[-2.0, -0.2, 0.0, 0.4, 1.8], [0.1, -0.1, 0.7, -0.8, 0.0]])
x = torch.randn(4, 5)

assert math.isclose(math.log2(3), 1.584962500721156, rel_tol=1e-12)
assert weight.shape == (2, 5)
assert x.shape[-1] == weight.shape[-1] == GROUP_SIZE


## 1. Absmean ternary：scale 与 code 分开保存

一种教学 recipe 用 `gamma=mean(abs(W))`，将 `W/gamma` round 后裁到 -1/0/1，再用 gamma 反量化。论文/实现的中心化、阈值和分组细节可能不同，必须把具体 recipe 写进制品。


In [ ]:
def ternary_quantize(weights, group_size, eps=1e-8):
    if weights.ndim != 2 or not isinstance(group_size, int) or group_size <= 0:
        raise ValueError("weights 必须为二维且 group_size 为正整数")
    codes_by_group, scales, dequantized_by_group = [], [], []
    for start in range(0, weights.shape[-1], group_size):
        group = weights[:, start:start + group_size]
        scale = group.abs().mean(-1, keepdim=True).clamp_min(eps)
        codes = torch.round(group / scale).clamp(-1, 1).to(torch.int8)
        codes_by_group.append(codes)
        scales.append(scale)
        dequantized_by_group.append(codes.float() * scale)
    return torch.cat(codes_by_group, -1), torch.stack(scales, 1), torch.cat(dequantized_by_group, -1)

# code 只含三值；每个输出通道/输入分组恰有一个 scale；零权重保持零。
codes, weight_scale, dequantized = ternary_quantize(weight, GROUP_SIZE)
assert set(codes.unique().tolist()) <= {-1, 0, 1}
assert dequantized.shape == weight.shape
assert weight_scale.shape == (weight.shape[0], math.ceil(weight.shape[1] / GROUP_SIZE), 1)
assert torch.equal(dequantized[weight == 0], torch.zeros_like(dequantized[weight == 0]))


## 2. Activation absmax int8：权重 1.58-bit 不等于整个算子 1.58-bit

输入激活仍需表示；可按 token 用 absmax 映射到 int8，再反量化参与教学计算。真实整数/混合精度数据流、accumulator 位宽和 scale 融合决定 kernel。


In [ ]:
def quantize_activation(activation, bits=8, eps=1e-8):
    if not isinstance(bits, int) or not 2 <= bits <= 8:
        raise ValueError("int8 容器只支持 2 到 8 bit 激活量化")
    qmax = 2 ** (bits - 1) - 1
    scale = activation.abs().amax(dim=-1, keepdim=True).clamp_min(eps) / qmax
    quantized = torch.round(activation / scale).clamp(-qmax, qmax).to(torch.int8)
    dequantized = quantized.float() * scale
    activation_ste = activation + (dequantized - activation).detach()
    return quantized, scale, activation_ste

# forward 数值为反量化值；STE backward 对每个输入元素传递单位梯度。
activation_probe = x.clone().requires_grad_(True)
act_q, act_scale, act_dq = quantize_activation(activation_probe)
assert act_q.dtype == torch.int8
assert act_scale.shape == (x.shape[0], 1)
assert torch.allclose(act_dq.detach(), act_q.float() * act_scale.detach())
act_dq.sum().backward()
assert torch.equal(activation_probe.grad, torch.ones_like(activation_probe))

# 反例边界 fail-closed：1 bit 会产生 qmax=0，超过 8 bit 会溢出 int8，二者都必须拒绝。
invalid_bits_rejected = []
for invalid_bits in (1, 9):
    try:
        quantize_activation(x, bits=invalid_bits)
    except ValueError:
        invalid_bits_rejected.append(invalid_bits)
assert invalid_bits_rejected == [1, 9]


## 3. STE：forward 用三值，backward 更新高精度 master weight

round 几乎处处梯度为零。Straight-Through Estimator 用 `W + (Q(W)-W).detach()`，forward 数值等于量化权重，backward 把梯度近似传给 W。它是有偏估计，但实用。


In [ ]:
def ternary_ste(weights, group_size):
    _, _, quantized = ternary_quantize(weights, group_size)
    return weights + (quantized - weights).detach()

# 权重 STE forward 等于分组反量化值，backward 更新每个浮点 master weight。
master = weight.clone().requires_grad_(True)
ste_weight = ternary_ste(master, GROUP_SIZE)
assert torch.allclose(ste_weight, ternary_quantize(master, GROUP_SIZE)[2])
ste_weight.sum().backward()
assert torch.allclose(master.grad, torch.ones_like(master))
assert master.dtype == torch.float32


## 4. 手写 BitLinear.forward：master、权重 code 与激活 scale 各司其职

模块参数仍是浮点 master weight；forward 对输入量化、对权重三值化，再做线性变换。bias 通常可省略或保持高精度。这里显式返回量化元数据用于测试，生产 API 可隐藏。


In [ ]:
class BitLinear(nn.Module):
    def __init__(self, in_features, out_features, group_size):
        super().__init__()
        if group_size <= 0:
            raise ValueError("group_size 必须为正")
        self.in_features, self.group_size = in_features, group_size
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.normal_(self.weight, std=1 / math.sqrt(in_features))

    def forward(self, inputs):
        if inputs.shape[-1] != self.in_features:
            raise ValueError("输入末维与 in_features 不一致")
        activation_codes, activation_scale, activation = quantize_activation(inputs)
        weight_for_forward = ternary_ste(self.weight, self.group_size)
        output = activation @ weight_for_forward.T
        return output, activation_codes, activation_scale

# 主 forward 同时使用激活/权重 STE；输入与 master weight 都收到有限非零梯度。
layer = BitLinear(5, 3, GROUP_SIZE)
train_input = x.clone().requires_grad_(True)
bit_output, input_codes, input_scale = layer(train_input)
bit_output.square().mean().backward()
assert bit_output.shape == (4, 3)
assert layer.weight.grad is not None and torch.isfinite(layer.weight.grad).all()
assert train_input.grad is not None and torch.isfinite(train_input.grad).all()
assert (train_input.grad.norm(dim=-1) > 0).all()
assert list(dict(layer.named_parameters())) == ["weight"]


## 5. 整数点积等价：先算 code，再合并 scale

若 activation 每行 scale 为 `s_x`、权重共享 scale 为 `s_w`，则反量化点积等于 `int_dot * s_x*s_w`。真实实现用更宽 accumulator 防溢出，并按分组/通道处理 scale。


In [ ]:
def grouped_integer_linear(input_codes, input_scale, weight_codes, weight_scales, group_size):
    output = torch.zeros(input_codes.shape[0], weight_codes.shape[0], dtype=torch.float32)
    accumulators = []
    for group_id, start in enumerate(range(0, input_codes.shape[-1], group_size)):
        stop = min(start + group_size, input_codes.shape[-1])
        accumulator = input_codes[:, start:stop].to(torch.int32) @ weight_codes[:, start:stop].to(torch.int32).T
        output += accumulator.float() * input_scale * weight_scales[:, group_id, 0][None, :]
        accumulators.append(accumulator)
    return output, accumulators

# 整数 oracle 直接复原主 BitLinear.forward，而不是另造一个无关 reference。
weight_codes, grouped_scale, _ = ternary_quantize(layer.weight.detach(), layer.group_size)
integer_reconstructed, accumulators = grouped_integer_linear(input_codes, input_scale, weight_codes, grouped_scale, layer.group_size)
assert torch.allclose(integer_reconstructed, bit_output.detach(), atol=1e-5)
assert all(accumulator.dtype == torch.int32 for accumulator in accumulators)
assert integer_reconstructed.shape == bit_output.shape


## 6. Base-3 位打包：理论 1.58 bit 需要真实编码才能兑现

把 -1/0/1 映射到 0/1/2，5 个 trit 可装进一个 byte，因为 `3^5=243<256`。这平均 1.6 bit/weight，接近信息下界；还需保存 scale、shape、尾部长度和对齐。


In [ ]:
def pack_trits(ternary_values):
    array = np.asarray(ternary_values, dtype=np.int8).reshape(-1)
    if not np.isin(array, [-1, 0, 1]).all():
        raise ValueError("只能打包 -1/0/1")
    digits = (array + 1).tolist()
    packed = bytearray()
    for start in range(0, len(digits), 5):
        value = sum(digit * (3 ** offset) for offset, digit in enumerate(digits[start:start + 5]))
        packed.append(value)
    return bytes(packed), len(digits)

def unpack_trits(packed, count):
    if not isinstance(count, int) or count < 0 or len(packed) != math.ceil(count / 5):
        raise ValueError("packed 长度与 count 不一致")
    raw_digits = []
    for byte in packed:
        if byte >= 3 ** 5:
            raise ValueError("存在非规范 base-3 byte")
        value = byte
        for _ in range(5):
            raw_digits.append(value % 3); value //= 3
    if any(raw_digits[count:]):
        raise ValueError("尾部未使用 trit 必须为规范零填充")
    return np.array(raw_digits[:count], dtype=np.int8) - 1

def pack_checkpoint(weight_codes, scales, group_size, activation_bits=8):
    if weight_codes.ndim != 2 or group_size <= 0 or not 2 <= activation_bits <= 8:
        raise ValueError("checkpoint 量化参数非法")
    expected_scale_shape = (weight_codes.shape[0], math.ceil(weight_codes.shape[1] / group_size), 1)
    if tuple(scales.shape) != expected_scale_shape or not torch.isfinite(scales).all() or not (scales > 0).all():
        raise ValueError("checkpoint scale shape/value 与分组合同不一致")
    packed, count = pack_trits(weight_codes.cpu().numpy())
    return {
        "packed": packed, "count": count, "shape": tuple(weight_codes.shape),
        "scales": scales.detach().clone(), "group_size": group_size,
        "activation_bits": activation_bits, "layout": "out-channel-contiguous-v1",
    }

def restore_checkpoint(checkpoint):
    required = {"packed", "count", "shape", "scales", "group_size", "activation_bits", "layout"}
    if not required <= set(checkpoint):
        raise ValueError("checkpoint 缺少必要元数据")
    shape, group_size = checkpoint["shape"], checkpoint["group_size"]
    if not isinstance(shape, (tuple, list)) or len(shape) != 2 or not isinstance(group_size, int) or group_size <= 0:
        raise ValueError("checkpoint shape/group_size 非法")
    if checkpoint["layout"] != "out-channel-contiguous-v1" or not 2 <= checkpoint["activation_bits"] <= 8:
        raise ValueError("checkpoint layout/activation_bits 不受支持")
    expected_scale_shape = (shape[0], math.ceil(shape[1] / group_size), 1)
    scales = checkpoint["scales"]
    if math.prod(shape) != checkpoint["count"] or tuple(scales.shape) != expected_scale_shape:
        raise ValueError("checkpoint count/scale shape 与权重布局不一致")
    if not torch.isfinite(scales).all() or not (scales > 0).all():
        raise ValueError("checkpoint scale 必须为有限正数")
    values = unpack_trits(checkpoint["packed"], checkpoint["count"])
    return torch.from_numpy(values.reshape(shape)), scales.clone(), group_size, checkpoint["activation_bits"]

def packed_checkpoint_linear(checkpoint, activation_codes, activation_scale):
    weight_codes, scales, group_size, activation_bits = restore_checkpoint(checkpoint)
    if activation_codes.dtype != torch.int8 or activation_scale.shape != (activation_codes.shape[0], 1):
        raise ValueError("activation code/scale 布局非法")
    qmax = 2 ** (activation_bits - 1) - 1
    if activation_codes.abs().max() > qmax or not torch.isfinite(activation_scale).all() or not (activation_scale > 0).all():
        raise ValueError("activation code/scale 数值非法")
    return grouped_integer_linear(activation_codes, activation_scale, weight_codes, scales, group_size)[0]

# packed checkpoint 可直接恢复并执行整数推理；坏 scale/布局/byte/count 均 fail-closed。
checkpoint = pack_checkpoint(weight_codes, grouped_scale, layer.group_size, activation_bits=8)
restored_codes, restored_scales, restored_group_size, restored_bits = restore_checkpoint(checkpoint)
checkpoint_output = packed_checkpoint_linear(checkpoint, input_codes, input_scale)
assert torch.equal(restored_codes, weight_codes.cpu()) and torch.equal(restored_scales, grouped_scale)
assert restored_group_size == layer.group_size and restored_bits == 8
assert torch.allclose(checkpoint_output, bit_output.detach(), atol=1e-5)

bad_scale = dict(checkpoint); bad_scale["scales"] = torch.ones(1)
bad_layout = dict(checkpoint); bad_layout["layout"] = "unknown"
bad_count = dict(checkpoint); bad_count["count"] -= 1
fail_closed = 0
for operation in (
    lambda: pack_trits([0, 2]), lambda: unpack_trits(bytes([255]), 1),
    lambda: restore_checkpoint(bad_scale), lambda: restore_checkpoint(bad_layout), lambda: restore_checkpoint(bad_count),
):
    try:
        operation()
    except ValueError:
        fail_closed += 1
assert fail_closed == 5


## 7. 误差与存储：必须把 scale/布局开销计入

三值误差可用输出相对误差与下游 loss 衡量。存储估计不能简单写参数数×1.58，还要计每组 scale、padding、索引和未量化 embedding/norm；训练还保留 master、梯度和优化器状态。


In [ ]:
def packed_storage_bytes(num_weights, group_size, scale_bytes=2):
    if num_weights < 0 or group_size <= 0:
        raise ValueError("容量参数非法")
    code_bytes = math.ceil(num_weights / 5)
    scale_count = math.ceil(num_weights / group_size)
    return code_bytes + scale_count * scale_bytes

# 估算使用与 forward 相同的分组合同；推理紧凑存储小于 FP16，训练 master 并未消失。
num_weights = 1_000_000
ternary_bytes = packed_storage_bytes(num_weights, group_size=GROUP_SIZE)
fp16_bytes = num_weights * 2
relative_error = (dequantized - weight).norm() / weight.norm()
assert ternary_bytes < fp16_bytes
assert 0 <= relative_error < 1
assert packed_storage_bytes(10, GROUP_SIZE) == 6


## 8. 发布门禁：从头训练 recipe 与 kernel layout 一起版本化

b1.58 的论文结果来自匹配的训练 recipe，不意味着任意 FP checkpoint 直接 ternary PTQ 仍保质量。manifest 绑定中心化/scale/阈值、激活位宽、分组、打包序、accumulator、kernel 与未量化层。


In [ ]:
@dataclass(frozen=True)
class BitNetArtifact:
    weight_scheme: str
    activation_bits: int
    group_size: int
    packing: str
    accumulator: str
    trained_with_quantization: bool

def artifact_hash(artifact):
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()

# artifact 的 group_size 与真实 BitLinear.forward、整数 oracle 和容量估算完全一致。
artifact = BitNetArtifact("grouped-absmean-ternary-v2", 8, GROUP_SIZE, "base3-5trits-canonical-v2", "int32", True)
digest = artifact_hash(artifact)
assert artifact.trained_with_quantization
assert artifact.group_size == layer.group_size
assert len(digest) == 64
assert digest != artifact_hash(BitNetArtifact(artifact.weight_scheme, 8, 4, artifact.packing, "int32", True))


## 面试收束：从公式走到生产合同

建议用六步回答：目标与约束、张量/数据合同、核心公式、正确性反例、质量—成本评测、版本与回滚。Notebook 的小模型只证明机制和边界，不代表论文规模结果、真实 GPU kernel 加速或线上泛化。生产替换时仍应保留同一批 oracle，并补齐目标硬件 profiling、分布式一致性、数据 provenance、安全审计和灰度发布。

继续追问时要主动区分：训练期方法与已有 checkpoint 的后处理、理论 FLOPs 与 wall-clock、平均质量与关键 slice、可逆近似与不可逆状态、模型置信与校准后的决策概率。
